# 06.2 — From GitHub to hosting

[Class index](README.md) · [Previous — 06.1](06.1-a-thin-local-interface.ipynb) · [Next — 06.3](06.3-automated-checks-and-updates.ipynb)

**Purpose:** Connect code in your own repository to a hosted Python process.

**Outcomes:** Prepare the root layout; select repository/branch/entrypoint/Python; identify a failed build stage; distinguish saved diagnostics from hosted evidence.

**Prerequisites:** The supplied files and setup, your GitHub/hosting accounts, and the familiar local run command. No previous exercise answer is used.

**In class — 30–50 minutes teaching/demo; 50–60 exercise. Independent work — 10 minutes.** The instructor starts a deployment while you inspect a supplied failure. Actual build time can overlap the diagnostic task; do not add it to the workload.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
required = ["streamlit_app.py", "pyproject.toml", "uv.lock", "fixtures", "solutions"]
if not all((root / name).exists() for name in required):
    raise RuntimeError(
        "Open 06-deployment.code-workspace from the complete class folder; "
        "select that folder's .venv kernel and restart. Expected app/fixtures here."
    )
print("Working folder:", root)
print("Kernel:", sys.executable)

In [ ]:
import json
from report_app.source import read_fixture

entrypoint = root / "streamlit_app.py"
assert entrypoint.is_file()
assert (root / "pyproject.toml").is_file() and (root / "uv.lock").is_file()
cases = read_fixture("deployment-failures.json")
print(json.dumps(cases[0], indent=2))

## Repository, branch, runtime, URL

A **repository** stores source/history. A **branch** identifies the stream of commits hosting should follow. A **runtime** is the environment executing the code, including its Python interpreter and installed packages. A **build** prepares that environment; a deployment starts/updates the hosted application. The public **URL** is the address a peer can open.

Copy this folder's contents into a separate student repository root. The course repository stores multiple classes under `classes/`; the student repository contains one class at its root. This makes `streamlit_app.py`, `pyproject.toml`, `uv.lock`, and `.github/workflows/checks.yml` unambiguous. A course fork with the workflow still nested under a class directory does not activate that workflow.

Follow the manual commands in [DEPLOYMENT.md](DEPLOYMENT.md), including inspection of staged files. Do not run `git init` inside the course folder, commit `.venv` or secrets, or force-push to work around an authentication error. GitHub browser/VS Code sign-in is separate from Python installation. Repository authorization must permit Streamlit to access the intended student repository.

### Visual guide — Copy contents into the student repository root

<img src="images/student-repository-root.png" alt="The nested course class contents are copied into a standalone student repository with streamlit_app, dependency files, and hidden GitHub workflow at its root." width="784" style="max-width: 100%; height: auto; background-color: white;">

The course keeps this class nested under classes/06-deployment. Your separate student repository puts the class contents at its root, including hidden workflow/configuration files. Exclude environments, caches, and real secrets.

*Credit: Original ADPRO course diagram (2026), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Editable Mermaid source](images/student-repository-root.mmd); [sources and teaching notes](IMAGE_SUGGESTIONS.md).*

## Choose the hosting inputs explicitly

In Community Cloud choose **Create app**, the existing-app option, your student repository, branch `main`, and entrypoint `streamlit_app.py`. In **Advanced settings**, choose **Python 3.13** to match the supplied `.python-version` and manifest. Leave AI secrets empty. Start the build and inspect its log rather than guessing from a loading spinner.

The hosting service recognizes `uv.lock`; keep the matching manifest with it. Do not add a competing `requirements.txt` to fix an unrelated entrypoint problem. A new package should be declared using uv and recorded in both changed metadata/lock files. The root layout also matches the hosting working directory. Our snapshot loader resolves paths from its module file, so it does not rely on an earlier notebook cell.

The current documented route is supported; an actual successful build still needs its own evidence. [Hosting steps](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/deploy), [dependency handling](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/app-dependencies).

In [ ]:
# Inspect the declared interpreter and selected runtime packages, without installing.
import tomllib
metadata = tomllib.loads((root / "pyproject.toml").read_text())
print("Selected minor:", (root / ".python-version").read_text().strip())
print("Allowed Python:", metadata["project"]["requires-python"])
print("Runtime packages:", metadata["project"]["dependencies"])

## Read the first useful error

A failed start can be caused by a wrong **entrypoint**, a missing **dependency**, the wrong **Python version**, or an application/data failure. State the stage and smallest plausible cause before changing files. For `File does not exist: app.py`, the supplied filename is the key fact. Installing extra packages will not create that file.

The JSON cases are explicitly invented teaching logs. They contain a stage, cause, proposed small fix, and next check. They let you practice while a real build completes; they do not demonstrate actual hosting, availability, or permissions. In real hosting logs, record only the necessary redacted line and the next actual result.

GitHub Pages serves static content, so publishing the notebook HTML there would not run our Python app. GitHub provides code and CI; Community Cloud runs the Streamlit server. The laptop's Ollama localhost is also not reachable through a remote server's localhost.

## Exercise 06.2.E1 — Choose the correct entrypoint

> **In class — 10 minutes**
>
> **Goal:** Diagnose a concrete start failure without unrelated edits.
>
> **Start here:** The entrypoint case printed above and streamlit_app.py in the supplied root.
>
> **Task:** Complete one filename string, state the cause, and select that entrypoint in your deployment if access is ready. While a build runs, inspect the saved dependency/Python cases.
>
> **Check:** The filename exists locally; record an actual build/URL result separately, or mark it pending.
>
> **Hint:** The app is supplied; correct the hosting file selection rather than creating app.py.

In [ ]:
# entrypoint_name = ...
# Cause:
# Actual hosting result or pending reason:

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_06_2_e1.py

## Independent work — 10 minutes

Continue [deployment guide sections 2–3](DEPLOYMENT.md). Check the selected repository/branch and Python minor. Inspect the build log and try the public URL with no AI secrets. Record what actually happened; if not ready, name the stage and continue the allocated deployment work in 06.4. Do not describe a saved log as a deployment.

**Completion check:** identify one error stage and next check. Explain why missing `pydantic` differs from a nonexistent entrypoint. **Reflection:** which evidence distinguishes a successful local run from a working public app?

Further setup/troubleshooting is inside the class's 60 deployment minutes, not an extra assignment. [Account authorization](https://docs.streamlit.io/deploy/streamlit-community-cloud/get-started/connect-your-github-account).